<a href="https://colab.research.google.com/github/Aosll/c-program/blob/New-features/Python_to_do_list.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================
# TASK MANAGEMENT APPLICATION (To-Do List)
# Name & Surname: Omer Efe Dikici
# =============================================================
# Features:
#   - Add, list, edit, and delete tasks
#   - Save and load tasks from a text file (utf-8)
#   - Error handling (empty input, invalid number, type error)
# =============================================================

import os
from datetime import datetime

FILE_NAME = "tasks.txt"  # File where tasks are stored

# Priority levels
PRIORITY_LEVELS = {"1": "High", "2": "Medium", "3": "Low"}


# ─────────────────────────────────────────────
# FILE OPERATIONS
# ─────────────────────────────────────────────

def load_tasks():
    tasks = []
    try:
        with open(FILE_NAME, "r", encoding="utf-8") as file:
            for line in file:
                line = line.strip()
                if line:  # Skip empty lines
                    parts = line.split("|")
                    # Backward compatibility: older format may contain only text
                    task = {
                        "text":      parts[0] if len(parts) > 0 else "",
                        "completed": parts[1] == "True" if len(parts) > 1 else False,
                        "priority":  parts[2] if len(parts) > 2 else "Medium",
                        "due_date":  parts[3] if len(parts) > 3 else "-"
                    }
                    if task["text"]:  # Skip if text is empty
                        tasks.append(task)
        print("Tasks loaded successfully.")
    except FileNotFoundError:
        print("Task file not found or could not be read. A new list has been created.")
    except Exception as error:
        print(f"An error occurred while reading the file: {error}. A new list has been created.")
    return tasks


def save_tasks(tasks):
    try:
        with open(FILE_NAME, "w", encoding="utf-8") as file:
            for task in tasks:
                line = f"{task['text']}|{task['completed']}|{task['priority']}|{task['due_date']}\n"
                file.write(line)
        print("Tasks saved successfully.")
    except Exception as error:
        print(f"An error occurred while saving the file: {error}")


# ─────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────

def print_task(index, task):
    status   = "✓" if task["completed"] else "○"
    priority = task["priority"]
    due_date = task["due_date"]
    print(f"  {index}. [{status}] {task['text']}  |  Priority: {priority}  |  Due Date: {due_date}")


def display_task_list(tasks):
    print("\n--- TASK LIST ---")
    if not tasks:
        print("  No tasks found.")
    else:
        for i, task in enumerate(tasks, start=1):
            print_task(i, task)
    print("-----------------")


def get_valid_number(tasks, action):

    display_task_list(tasks)
    if not tasks:
        return None

    while True:
        try:
            number = int(input(f"Enter the number of the task you want to {action}: "))
            if 1 <= number <= len(tasks):
                return number - 1  # Convert to 0-based index
            else:
                print("Invalid task number!")
        except ValueError:
            print("Please enter a number!")


# ─────────────────────────────────────────────
# CORE FUNCTIONS
# ─────────────────────────────────────────────

def list_tasks(tasks):

    display_task_list(tasks)


def add_task(tasks):

    text = input("New task: ").strip()
    if not text:
        print("Task text cannot be empty!")
        return

    print("Select priority level: 1) High  2) Medium  3) Low  (Default: Medium)")
    priority_choice = input("Priority (1/2/3, leave blank=Medium): ").strip()
    priority = PRIORITY_LEVELS.get(priority_choice, "Medium")


    due_date = input("Due date (DD.MM.YYYY, leave blank=none): ").strip()
    if due_date:
        try:
            # Validate the date format
            datetime.strptime(due_date, "%d.%m.%Y")
        except ValueError:
            print("Invalid date format. Due date has been set to '-'.")
            due_date = "-"
    else:
        due_date = "-"

    new_task = {
        "text":      text,
        "completed": False,
        "priority":  priority,
        "due_date":  due_date
    }
    tasks.append(new_task)
    print(f"Task '{text}' has been added.")
    save_tasks(tasks)


def edit_task(tasks):

    index = get_valid_number(tasks, "edit")
    if index is None:
        return

    old_text = tasks[index]["text"]
    new_text = input(f"New task text ({old_text}): ").strip()

    if not new_text:
        print("Task text cannot be empty! Edit cancelled.")
        return

    tasks[index]["text"] = new_text
    print("Task updated successfully.")
    save_tasks(tasks)


def delete_task(tasks):

    index = get_valid_number(tasks, "delete")
    if index is None:
        return

    deleted = tasks.pop(index)
    print(f"Task '{deleted['text']}' has been deleted.")
    save_tasks(tasks)

def toggle_task_completion(tasks):

    index = get_valid_number(tasks, "toggle status of")
    if index is None:
        return

    tasks[index]["completed"] = not tasks[index]["completed"]
    status = "completed ✓" if tasks[index]["completed"] else "not completed ○"
    print(f"Task '{tasks[index]['text']}' has been marked as {status}.")
    save_tasks(tasks)


def sort_tasks(tasks):

    print("\nSelect a sorting criterion:")
    print("  1) By priority (High → Low)")
    print("  2) By due date (Soonest → Latest)")
    print("  3) By status (Incomplete → Complete)")

    try:
        criterion = int(input("Your choice (1-3): "))
    except ValueError:
        print("Invalid choice!")
        return

    priority_order = {"High": 1, "Medium": 2, "Low": 3}

    if criterion == 1:
        tasks.sort(key=lambda t: priority_order.get(t["priority"], 99))
        print("Tasks sorted by priority.")
    elif criterion == 2:
        def date_key(t):
            try:
                return datetime.strptime(t["due_date"], "%d.%m.%Y")
            except ValueError:
                return datetime.max  # Tasks without a date go to the end
        tasks.sort(key=date_key)
        print("Tasks sorted by due date.")
    elif criterion == 3:
        tasks.sort(key=lambda t: t["completed"])
        print("Tasks sorted by completion status.")
    else:
        print("Invalid choice!")
        return

    save_tasks(tasks)


# ─────────────────────────────────────────────
# MENU
# ─────────────────────────────────────────────

def show_menu():
    """Prints the main menu to the screen."""
    print("\n--- TO-DO LIST APPLICATION ---")
    print("  1. List Tasks")
    print("  2. Add New Task")
    print("  3. Edit Task")
    print("  4. Delete Task")
    print("  5. Toggle Task Status    [Advanced]")
    print("  6. Sort Tasks            [Advanced]")
    print("  7. Exit")


def main_menu():

    print("\nWelcome to the To-Do List Application!")
    tasks = load_tasks()

    # Menu option → function mapping
    actions = {
        "1": lambda: list_tasks(tasks),
        "2": lambda: add_task(tasks),
        "3": lambda: edit_task(tasks),
        "4": lambda: delete_task(tasks),
        "5": lambda: toggle_task_completion(tasks),
        "6": lambda: sort_tasks(tasks),
    }

    while True:
        show_menu()
        choice = input("Your choice (1-7): ").strip()

        if choice == "7":
            print("Exiting the program...")
            break
        elif choice in actions:
            actions[choice]()
        else:
            print("Invalid choice! Please enter a value between 1 and 7.")


# ─────────────────────────────────────────────
# PROGRAM ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    main_menu()